## Practica Primer Parcial
Nombre: Santiago Nuñez Perez

### Objetivo: Establecer una relación entre los comentarios de usuarios de una biblioteca pública y una combinación lineal de las siguientes valoraciones:

Adecuación de la colección,
Estado del material,
Organización,
Ordenación,
Trato,
Información,
Horario,
WiFi,
Estado,
Limpieza y
Valoración global.
Instrucciones:

Descargue el archivo data_biblio.csvDownload data_biblio.csv
Para las variables numéricas, sustituya los datos faltantes basados en el tipo de variable
Codificar los comentarios (<5 CAUSA) usando TfIdf.
Criterio para NaN: Imputación por una constante (una palabra o palabras que identifiquen la muestra vacía).
Reducir la dimensionalidad de la codificación a usando como función hash con

In [11]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression

# 1. Cargar el conjunto de datos
df = pd.read_csv('C:/Users/santi/OneDrive - ITESO/ITESO/Semestre 3/Ingenieria de Caracteristicas/Ingenieria-De-Caracteristicas/Datos/datos_biblio.csv')

print('Dimensiones del DataFrame:', df.shape)
print('\nValores faltantes por columna:')
print(df.isna().sum())
df.head()

Dimensiones del DataFrame: (1095, 12)

Valores faltantes por columna:
ADECUACIÓN COLECCIÓN    132
ESTADO MATERIAL          92
ORGANIZACIÓN            100
ORDENACIÓN               99
TRATO                    28
INFORMACIÓN.1           187
HORARIO                  74
WIFI.1                  620
ESTADO                   31
LIMPIEZA                 19
<5 CAUSA                892
VALORACION GLOBAL         0
dtype: int64


,ADECUACIÓN COLECCIÓN,ESTADO MATERIAL,ORGANIZACIÓN,ORDENACIÓN,TRATO,INFORMACIÓN.1,HORARIO,WIFI.1,ESTADO,LIMPIEZA,<5 CAUSA,VALORACION GLOBAL
0,2.414739,2.736965,5.035845,7.117318,609.061849,7.291122,3.835400,1.559955,0.815098,18.645453,NaN,8.877004
1,NaN,4.462921,6.953253,14.388629,1256.317719,NaN,3.835400,NaN,0.991220,29.849461,NaN,8.877004
2,2.414739,4.462921,3.566761,4.838591,1256.317719,4.942623,3.835400,1.971928,0.815098,18.645453,NaN,4.326990
3,3.065006,2.736965,5.035845,10.223330,1256.317719,14.819607,1.377896,0.901784,0.659037,46.608423,NaN,8.877004
4,2.414739,3.524037,5.035845,10.223330,609.061849,10.501788,NaN,NaN,0.991220,29.849461,NaN,8.877004


## Paso 2: 
Imputación de datos faltantes en variables numéricasEn el conjunto de datos existen 11 variables numéricas (escalas continuas/ordinales con asimetrías y distribuciones particulares):

Para distribuciones numéricas con sesgo, valores atípicos o discretas/ordinales, el método más robusto visto en clase es la mediana.   Si alguna variable presenta distribución estrictamente simétrica y normal, se utiliza la media. 

In [12]:
# Lista de las 11 variables numéricas de valoración
cols_numericas = [
    'ADECUACIÓN COLECCIÓN',
    'ESTADO MATERIAL',
    'ORGANIZACIÓN',
    'ORDENACIÓN',
    'TRATO',
    'INFORMACIÓN.1',
    'HORARIO',
    'WIFI.1',
    'ESTADO',
    'LIMPIEZA',
    'VALORACION GLOBAL',
]

df_limpio = df.copy()

# Cálculo de asimetría para seleccionar el estimador óptimo
asimetrias = df_limpio[cols_numericas].skew()
print('Asimetría de las variables numéricas:\n', asimetrias)

for col in cols_numericas:
  # Criterio de simetría
  if abs(asimetrias[col]) > 0.5:
    valor_imputado = df_limpio[col].median()
    print(f'{col:22s} -> Imputada con Mediana: {valor_imputado:.4f}')
  else:
    valor_imputado = df_limpio[col].mean()
    print(f'{col:22s} -> Imputada con Media:   {valor_imputado:.4f}')

  df_limpio[col] = df_limpio[col].fillna(valor_imputado)

print(
    '\nValores nulos restantes en numéricas:',
    df_limpio[cols_numericas].isna().sum().sum(),
)

Asimetría de las variables numéricas:
 ADECUACIÓN COLECCIÓN   -0.082769
ESTADO MATERIAL        -0.119039
ORGANIZACIÓN           -0.269861
ORDENACIÓN             -0.263597
TRATO                  -0.705079
INFORMACIÓN.1          -0.283157
HORARIO                -0.337884
WIFI.1                 -0.220784
ESTADO                  0.523634
LIMPIEZA               -0.445399
VALORACION GLOBAL      -0.115203
dtype: float64
ADECUACIÓN COLECCIÓN   -> Imputada con Media:   2.3892
ESTADO MATERIAL        -> Imputada con Media:   2.8371
ORGANIZACIÓN           -> Imputada con Media:   6.0265
ORDENACIÓN             -> Imputada con Media:   9.4099
TRATO                  -> Imputada con Mediana: 1256.3177
INFORMACIÓN.1          -> Imputada con Media:   9.7928
HORARIO                -> Imputada con Media:   4.2661
WIFI.1                 -> Imputada con Media:   1.7711
ESTADO                 -> Imputada con Mediana: 0.9912
LIMPIEZA               -> Imputada con Media:   31.5660
VALORACION GLOBAL      -> Imp

## Paso 3
Codificación TF-IDF de los comentarios (<5 CAUSA) con imputación constante
Se imputan las muestras vacías (NaN) mediante una cadena constante (por ejemplo, 'sin_comentario') y posteriormente se extraen las ponderaciones TF-IDF

In [14]:
col_texto = '<5 CAUSA'

# Criterio para NaN: Imputación por una constante que identifica la muestra vacía
df_limpio[col_texto] = df_limpio[col_texto].fillna('sin_comentario')

# Codificación TfIdf
vectorizador_tfidf = TfidfVectorizer()
X_tfidf = vectorizador_tfidf.fit_transform(df_limpio[col_texto])

vocabulario = vectorizador_tfidf.get_feature_names_out()
n = len(vocabulario)

print(f'Número de observaciones (N): {X_tfidf.shape[0]}')
print(
    f'Dimensión del espacio original TF-IDF (n = tamaño del vocabulario): {n}'
)

Número de observaciones (N): 1095
Dimensión del espacio original TF-IDF (n = tamaño del vocabulario): 1018


## Paso 4:
Reducción de dimensionalidad mediante Feature Hashing (multiplicación de matrices)

Dado un vector de comentario $x \in \mathbb{R}^{n \times 1}$, la proyección hash lineal $f_h(x) = Ax$ requiere construir la matriz hash $A \in \mathbb{R}^{m \times n}$.
Cada columna $j$ de $A$ (correspondiente al término $j$ del vocabulario) asigna un 1 en la fila $i = h(\text{palabra}_j) \pmod m$:   Para la matriz completa de datos $X$ (cuyas filas son las observaciones $x^T$), la transformación matricial equivale a:$$X_{\text{hash}} = X \cdot A^T \in \mathbb{R}^{N \times m}$$

In [16]:
# Dimensión objetivo m (puedes ajustar m=1 para una relación unidimensional directa, o m>1)
m = 1

# Construcción de la matriz A de tamaño (m, n)
A = np.zeros((m, n))

for j, palabra in enumerate(vocabulario):
  # Función hash sobre cada término j proyectada al espacio {0, ..., m-1}
  # Usao de abs(hash(palabra)) para garantizar índices enteros positivos
  indice_m = abs(hash(palabra)) % m
  A[indice_m, j] = 1.0

print(f'Dimensiones de la matriz Hash A: {A.shape} (m = {m}, n = {n})')

# Aplicación de fh(x) = A @ x para todo el conjunto:
# Como cada fila de X_tfidf es x^T, la multiplicación es X_tfidf @ A.T
X_reducido = X_tfidf.dot(A.T)

print('Dimensiones antes de reducción (TF-IDF):', X_tfidf.shape)
print('Dimensiones después de reducción (Hashing):', X_reducido.shape)

Dimensiones de la matriz Hash A: (1, 1018) (m = 1, n = 1018)
Dimensiones antes de reducción (TF-IDF): (1095, 1018)
Dimensiones después de reducción (Hashing): (1095, 1)


In [18]:
# Variables independientes: las 11 valoraciones numéricas
X_val = df_limpio[cols_numericas].values

# Variable dependiente: la representación hash reducida de los comentarios
y_comentarios = X_reducido if m > 1 else X_reducido.ravel()

# Ajuste del modelo de regresión lineal
modelo = LinearRegression()
modelo.fit(X_val, y_comentarios)

# Coeficiente de determinación R^2
r2 = modelo.score(X_val, y_comentarios)

print('COMBINACIÓN LINEAL OBTENIDA')
print(f'Coeficiente de determinación R^2: {r2:.6f}')
print(f'Término independiente (Intercepto / beta_0): {modelo.intercept_}')
print('\nPesos de la combinación lineal para cada valoración (beta_1 a beta_11):')

if m == 1:
  for col, coef in zip(cols_numericas, modelo.coef_):
    print(f'  {col:22s}: {coef:12.6f}')
else:
  df_coefs = pd.DataFrame(modelo.coef_, columns=cols_numericas)
  df_coefs.index = [f'Dim_Hash_{i}' for i in range(m)]
  print(df_coefs)

COMBINACIÓN LINEAL OBTENIDA
Coeficiente de determinación R^2: 0.099518
Término independiente (Intercepto / beta_0): 2.3787224709162222

Pesos de la combinación lineal para cada valoración (beta_1 a beta_11):
  ADECUACIÓN COLECCIÓN  :    -0.056604
  ESTADO MATERIAL       :     0.067015
  ORGANIZACIÓN          :    -0.007197
  ORDENACIÓN            :     0.012944
  TRATO                 :    -0.000041
  INFORMACIÓN.1         :     0.032892
  HORARIO               :    -0.070551
  WIFI.1                :    -0.169652
  ESTADO                :    -0.389853
  LIMPIEZA              :     0.006062
  VALORACION GLOBAL     :    -0.077868
